In [ ]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# impoting kaggle key or api
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Mounted at /content/drive


In [ ]:
!kaggle datasets download -d mattimansha/inspirational-quotes

Dataset URL: https://www.kaggle.com/datasets/mattimansha/inspirational-quotes
License(s): CC-BY-SA-4.0
  0% 0.00/139k [00:00<?, ?B/s]
100% 139k/139k [00:00<00:00, 476MB/s]


In [ ]:
!unzip '/content/inspirational-quotes.zip'

Archive:  /content/inspirational-quotes.zip
  inflating: insparation.csv         


In [ ]:
import pandas as pd
df=pd.read_csv('/content/insparation.csv')
df.head()

,Unnamed: 0,Category,Quote,Image-link,Quote-url
0,0,LOVE,Let us see what love can do.,https://assets.passiton.com/quotes/quote_artwo...,/inspirational-quotes/6900-let-us-see-what-lov...
1,1,LOVE,We can’t heal the world today. But we can begi...,https://assets.passiton.com/quotes/quote_artwo...,/inspirational-quotes/8169-we-can-t-heal-the-w...
2,2,LISTENING,Listen with curiosity. Speak with honesty. Act...,https://assets.passiton.com/quotes/quote_artwo...,/inspirational-quotes/8083-listen-with-curiosi...
3,3,LISTENING,The most basic and powerful way to connect to ...,https://assets.passiton.com/quotes/quote_artwo...,/inspirational-quotes/7139-the-most-basic-and-...
4,4,LISTENING,"Knowledge speaks, but wisdom listens.",https://assets.passiton.com/quotes/quote_artwo...,/inspirational-quotes/8376-knowledge-speaks-bu...


In [ ]:
df = df[["Category", "Quote"]]
df = df.dropna()
df["Category"] = df["Category"].str.lower()


In [ ]:
df.head()

,Category,Quote
0,love,Let us see what love can do.
1,love,We can’t heal the world today. But we can begi...
2,listening,Listen with curiosity. Speak with honesty. Act...
3,listening,The most basic and powerful way to connect to ...
4,listening,"Knowledge speaks, but wisdom listens."


In [ ]:
text_data = []

for i in range(len(df)):
    category = df["Category"].iloc[i]
    quote = df["Quote"].iloc[i]
    combined = category + " " + quote
    text_data.append(combined.lower())

In [ ]:
cleaned_text = []

for line in text_data:
    line = re.sub(r'[^a-zA-Z\s]', '', line)  # remove punctuation
    cleaned_text.append(line)

In [ ]:
VOCAB_SIZE = 5000
MAX_SEQ_LEN = 25

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(cleaned_text)

print("Vocabulary size:", VOCAB_SIZE)

Vocabulary size: 5000


In [ ]:
input_sequences = []
for line in cleaned_text:
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram = token_list[:i+1]
        input_sequences.append(n_gram)
# Pad sequences
input_sequences = pad_sequences(
    input_sequences,
    maxlen=MAX_SEQ_LEN,
    padding='pre'
)
# Split into x and y
x = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("Total training samples:", len(x))

Total training samples: 35988


In [ ]:
model = Sequential()
model.add(Embedding(VOCAB_SIZE, 128, input_length = MAX_SEQ_LEN-1))
model.add(LSTM(256, return_sequences=True))
model.add(LSTM(128))
model.add(Dropout(0.4))
model.add(Dense(VOCAB_SIZE, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(monitor='loss', patience = 5)

history = model.fit(x, y, epochs=200, batch_size=128, callbacks=[early_stop])

Epoch 1/200
282/282 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - accuracy: 0.0401 - loss: 6.8446
Epoch 2/200
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.0530 - loss: 6.2384
Epoch 3/200
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.0528 - loss: 6.1199
Epoch 4/200
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.0612 - loss: 5.9777
Epoch 5/200
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.0719 - loss: 5.8739
Epoch 6/200
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.0842 - loss: 5.7483
Epoch 7/200
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.0889 - loss: 5.6631
Epoch 8/200
282/282 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.0999 - loss: 5.5666
Epoch 9/200
282/282 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.1057 - loss: 5.4684
Epoch 10/200
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.1121 - loss: 5.3849
Epoch 11/200
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.1189 - loss: 5.3096
Epoch 12/200
282/282 ━━━━━━━━

In [ ]:
# Save trained model
model.save("quote_model.h5")
# Save tokenizer
import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
print("Model and tokenizer saved.")

Model and tokenizer saved.


In [ ]:
from tensorflow.keras.models import load_model
# Load model
model = load_model("quote_model.h5")
# Load tokenizer
with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)
# Reverse mapping
index_to_word = {index: word for word, index in tokenizer.word_index.items()}
max_seq_len = 25

In [ ]:
def sample(preds, temperature=0.7):
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)

In [ ]:
def generate_quote(seed_text, next_words=25, temperature=0.7):
    seed_text = seed_text.lower()
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences(
            [token_list],
            maxlen=max_seq_len - 1,
            padding='pre'
        )
        predictions = model.predict(token_list, verbose=0)[0]
        predicted_index = sample(predictions, temperature)
        next_word = index_to_word.get(predicted_index)
        if not next_word:
            break
        seed_text += " " + next_word
    return seed_text

In [ ]:
print(generate_quote("success", next_words=25, temperature=0.7))

success for every day we can choose it for every light practice there is a great act of our children and have a thing that love


In [ ]:
print(generate_quote("love", next_words=25, temperature=0.7))

love my success of your dreams make a difference but nothing can be better to have it in them and do or any experience or know


In [ ]:
print(tf.__version__)

2.19.0


ValueError: Invalid filepath extension for saving. Please add either a `.keras` extension for the native Keras format (recommended) or a `.h5` extension. Use `model.export(filepath)` if you want to export a SavedModel for use with TFLite/TFServing/etc. Received: filepath=quote_model.pkl.